# S23-S24 · Brazo robótico: simulador y ventosa (DOBOT Magician)

**Objetivo (RA4-b/c/d):** programar los **comandos de movimiento**, el **dibujo** y el **pick & place con ventosa** usando una clase `BrazoSimulado` (el equivalente en Python al **simulador DobotLab**). Cubre las dos sesiones prácticas; en el aula el brazo lleva montada **solo la ventosa**.


## 1. El brazo simulado (`BrazoSimulado`)

`BrazoSimulado` imita la API real (`pydobot` / `dType`) y **valida los rangos**: X entre 150 y 320 mm, radio ≤ 320 mm, Z entre −80 y 140 mm. Registra las órdenes y dibuja la trayectoria. Igual que el **simulador DobotLab** (Basic scene), pero en Python.


In [ ]:
%matplotlib inline
import math
import matplotlib.pyplot as plt

class BrazoSimulado:
    """Imita la API de pydobot y valida el script SIN hardware."""
    X_MIN, X_MAX = 150.0, 320.0     # rango de X permitido
    RADIO_MAX = 320.0               # alcance del DOBOT Magician
    Z_MIN, Z_MAX = -80.0, 140.0     # limites de Z

    def __init__(self):
        self.pos = [200.0, 0.0, 50.0, 0.0]
        self.ventosa_activa = False
        self.traza = []

    def _validar(self, x, y, z, r):
        if not (self.X_MIN <= x <= self.X_MAX):
            raise ValueError(f"X={x} fuera de [{self.X_MIN}, {self.X_MAX}]")
        if math.hypot(x, y) > self.RADIO_MAX:
            raise ValueError(f"radio {math.hypot(x, y):.1f} > {self.RADIO_MAX}")
        if not (self.Z_MIN <= z <= self.Z_MAX):
            raise ValueError(f"Z={z} fuera de [{self.Z_MIN}, {self.Z_MAX}]")

    def move_to(self, x, y, z, r=0.0, wait=True):
        """Mueve la punta a (x, y, z) en mm (equivale a MoveL / MoveJ)."""
        self._validar(x, y, z, r)
        self.pos = [float(x), float(y), float(z), float(r)]
        self.traza.append(("move", float(x), float(y), float(z)))

    def suction(self, activar=True):
        """Activa/desactiva la ventosa (en pydobot: robot.suck(True/False))."""
        self.ventosa_activa = bool(activar)
        self.traza.append(("suction", activar))

    def wait(self, ms):
        self.traza.append(("wait", ms))

    def home(self):
        self.move_to(200, 0, 50, 0)

    def resumen(self):
        return f"ordenes={len(self.traza)} ventosa={'activa' if self.ventosa_activa else 'off'} pos={self.pos}"

    def dibujar_trayectoria(self, titulo="Trayectoria XY", z_corte=None):
        """Dibuja la trayectoria; si z_corte no es None, solo los puntos con z <= z_corte."""
        pts = [(t[1], t[2]) for t in self.traza if t[0] == "move" and (z_corte is None or t[3] <= z_corte)]
        plt.figure(figsize=(5, 5))
        if pts:
            xs, ys = zip(*pts)
            plt.plot(xs, ys, "-o", markersize=3)
        plt.scatter([0], [0], c="k", marker="x", label="base")
        plt.gca().add_patch(plt.Circle((0, 0), self.RADIO_MAX, fill=False, ls="--", color="gray"))
        plt.axis("equal"); plt.grid(True); plt.title(titulo)
        plt.xlabel("X (mm)"); plt.ylabel("Y (mm)"); plt.legend()
        plt.show()


## 2. Comandos de movimiento (MoveJ / MoveL / Jump)

- **MoveJ** — movimiento **articular** (cada eje a su velocidad). Rápido, trayectoria impredecible.
- **MoveL** — movimiento **lineal** en cartesianas. Recta, más lento.
- **Jump** — sube, se desplaza y baja en arco (útil para no arrastrar la pieza).

En `BrazoSimulado`, `move_to(x, y, z)` encadena puntos; un movimiento tipo *Jump* es subir → desplazarse → bajar.


In [ ]:
brazo = BrazoSimulado()

brazo.move_to(250, 0, 60)     # punto seguro (alto)
brazo.move_to(250, 0, 10)     # bajar a la pieza
brazo.wait(500)
brazo.move_to(250, 0, 60)     # subir de nuevo
brazo.move_to(280, 40, 60)    # desplazarse en alto
brazo.move_to(280, 40, 10)    # bajar en destino

print(brazo.resumen())
brazo.dibujar_trayectoria("Secuencia tipo Jump")


## 3. Dibujar (como «Escribir/Dibujar» del simulador)

En el **simulador DobotLab** se dibuja con **Escribir/Dibujar**. Aquí lo reproducimos en Python: mover con Z alto, bajar a Z de dibujo y recorrer los vértices.


In [ ]:
brazo = BrazoSimulado()
vertices = [(250, -25), (250, 25), (300, 25), (300, -25), (250, -25)]   # cuadrado

brazo.move_to(vertices[0][0], vertices[0][1], 20)   # posicionar arriba (Z=20)
for (x, y) in vertices:                              # recorrer con Z=0 (dibujando)
    brazo.move_to(x, y, 0)

print(brazo.resumen())
brazo.dibujar_trayectoria("Cuadrado (trazo con Z=0)", z_corte=0)


## 4. Ventosa: coger y soltar (pick & place)

En el brazo real solo hay **ventosa** (`suck(True/False)`), que **aspira** superficies lisas y no porosas. Secuencia canónica: *punto seguro → bajar → aspirar → esperar → subir → desplazar → bajar → soltar → subir*.


In [ ]:
brazo = BrazoSimulado()

brazo.move_to(240, 0, 60)     # pre-pick (alto)
brazo.move_to(240, 0, 10)     # pick (bajar a la pieza)
brazo.suction(True)           # aspirar
brazo.wait(500)               # dejar que haga vacío
brazo.move_to(240, 0, 60)     # subir con la pieza
brazo.move_to(240, 60, 60)    # pre-place (alto)
brazo.move_to(240, 60, 10)    # place (bajar)
brazo.suction(False)          # soltar
brazo.move_to(240, 60, 60)    # subir

print(brazo.resumen())
brazo.dibujar_trayectoria("Pick & place con ventosa")


## 5. La cinta transportadora (`CintaSimulada`)

El kit añade una **cinta** con **fotocélula** (detecta presencia) y **sensor de color**. `CintaSimulada` imita la cinta + fotocélula.


In [ ]:
class CintaSimulada:
    """Imita la cinta + fotocelula del kit DOBOT."""
    def __init__(self):
        self.encendida = False
        self.objeto = False
    def encender(self): self.encendida = True
    def apagar(self): self.encendida = False
    def hay_objeto(self): return self.objeto
    def llega_objeto(self): self.objeto = True


## 6. Pick & place desde la cinta

La cinta avanza; cuando la **fotocélula** detecta el objeto, la paramos, lo cogemos con la ventosa y lo dejamos en la zona marcada.


In [ ]:
brazo = BrazoSimulado()
cinta = CintaSimulada()

cinta.encender()
cinta.llega_objeto()                 # en el aula lo detecta la fotocelula

if cinta.hay_objeto():
    cinta.apagar()
    brazo.move_to(250, -40, 60)      # pre-pick
    brazo.move_to(250, -40, 0)       # pick
    brazo.suction(True)
    brazo.wait(500)
    brazo.move_to(250, -40, 60)      # subir
    brazo.move_to(250, 40, 60)       # pre-place
    brazo.move_to(250, 40, 0)        # place
    brazo.suction(False)
    brazo.wait(500)
    brazo.move_to(250, 40, 60)       # subir
    brazo.home()

print(brazo.resumen())
brazo.dibujar_trayectoria("Pick & place desde la cinta")


## 7. Paletización

El paletizado es un pick & place con **punto de cogida fijo** y **punto de dejada variable**: recorremos una **cuadrícula** (filas × columnas) con bucles.


In [ ]:
brazo = BrazoSimulado()

x0, y0 = 220, -40        # esquina del palé (mm)
paso_x, paso_y = 30, 30  # separación entre celdas (mm)
n_filas, n_columnas = 3, 2

for fila in range(n_filas):
    for col in range(n_columnas):
        x = x0 + col * paso_x
        y = y0 + fila * paso_y
        brazo.move_to(200, 0, 60)      # ir a coger (punto fijo)
        brazo.move_to(200, 0, 10)
        brazo.suction(True)
        brazo.move_to(200, 0, 60)
        brazo.move_to(x, y, 60)        # ir a la celda
        brazo.move_to(x, y, 10)
        brazo.suction(False)
        brazo.move_to(x, y, 60)

print(brazo.resumen())
brazo.dibujar_trayectoria("Paletizado 3x2")


## 8. (Optativa) Clasificación por color

El **sensor de color** devuelve `[R, G, B]` (`dType.GetColorSensor(api)`): `[1,0,0]` rojo, `[0,1,0]` verde, `[0,0,1]` azul. La actividad optativa: coger cada cubo, leer su color y depositarlo en una **zona distinta** según el color.


In [ ]:
class SensorColorSimulado:
    """Imita el sensor de color: en el aula devuelve [R,G,B] (dType.GetColorSensor)."""
    def __init__(self, color="rojo"):
        self.color = color
    def leer(self):
        return self.color

zonas = {"rojo": (240, 40), "verde": (240, 0), "azul": (240, -40)}   # zona de cada color

def clasificar(brazo, color):
    """Coge una pieza y la deposita en la zona de su color."""
    x, y = zonas[color]
    brazo.suction(True)          # aspirar (ya está sobre la pieza)
    brazo.wait(300)
    brazo.move_to(240, 0, 60)    # subir
    brazo.move_to(x, y, 60)      # ir a la zona (alto)
    brazo.move_to(x, y, 10)      # bajar
    brazo.suction(False)         # soltar
    brazo.move_to(x, y, 60)      # subir

brazo = BrazoSimulado()
sensor = SensorColorSimulado("verde")   # la pieza detectada es verde
clasificar(brazo, sensor.leer())

print(brazo.resumen())
brazo.dibujar_trayectoria("Clasificación por color (verde)")


## Actividad · Prueba el código y concluye

1. **Prueba el código** cambiando parámetros y volviendo a ejecutar:
   - **Movimiento/dibujo:** cambia puntos, alturas y la figura (`vertices`).
   - **Pick & place:** mueve la pieza a otro destino y ajusta alturas/tiempos.
   - **Paletización:** cambia `n_filas`/`n_columnas` y observa la cuadrícula.
   - **(Optativa) Clasificación:** prueba con `"rojo"` y `"azul"` y comprueba que cada color va a su zona.
2. **Prueba a fallar:** pide un punto con `X < 150` o radio > 320 y mira qué error lanza `BrazoSimulado`.
3. **Añade una conclusión de 3–4 líneas**: por qué conviene validar **sin el robot** (offline) y qué diferencia hay entre MoveJ y MoveL.
